# 第 3 章 线性模型（上）：Logistic 回归二分类

**分类**是机器学习中最常见的一类任务，预测标签是离散符号，也称为**类别**。根据类别数量，分类任务又可以分为二分类任务和多分类任务。

**线性分类**是指利用一个或多个线性函数加上决策规则将样本进行分类。所谓线性并不是说从输入特征到预测标签之间的映射是线性的，而是指在样本空间中不同类别的分界面是一个超平面。不同线性分类模型之间的区别主要是模型、损失函数和优化算法的不同。常用的线性分类模型有感知器、支持向量机、Logistic 回归和 Softmax 回归等。

- **Logistic 回归**：处理二分类问题的经典线性模型；
- **Softmax 回归**：Logistic 回归在多分类问题上的推广。

本节实现 Logistic 回归，对 Moon1000 数据集做二分类；下一节实现 Softmax 回归并做鸢尾花多分类。


## 1. Logistic 回归

Logistic 回归会将输入特征与权重做线性叠加，然后通过非线性函数 $g:\mathbb{R}^D \to (0,1)$ 输出后验概率 $p(y=1\mid \boldsymbol{x})$：

$$ p(y=1\mid \boldsymbol{x}) = \sigma(\boldsymbol{w}^\top \boldsymbol{x}+b), $$

其中 $\sigma(\cdot)$ 为 **Logistic 函数**（也叫 sigmoid 函数）：

$$ \sigma(x)=\frac{1}{1+\exp(-x)}. $$

下面先实现 Logistic 函数。


In [ ]:
import torch
import matplotlib.pyplot as plt


# 定义 Logistic 函数
def logistic(x):
    return 1 / (1 + torch.exp(-x))


**练一练**：调用上面定义的 `logistic` 函数，在 $[-10, 10]$ 区间内采样，并用 Matplotlib 画出它的函数曲线，观察“两端饱和、中间近似线性”的形状。


In [ ]:
# 练一练：在 [-10, 10] 区间画出 Logistic 函数曲线
# 提示：用 torch.linspace 采样，再调用 logistic(x)，最后用 plt.plot 绘制
x = torch.linspace(-10, 10, 10000)
# TODO: 在这里补全绘图代码


当输入在 0 附近时，Logistic 函数近似为线性函数。而当输入值非常大或非常小时，函数会对输入进行抑制。输入越小，越接近 0；输入越大，越接近 1。正因为 Logistic 函数具有这样的性质，使得其输出可以直接看作概率。


## 2. 构建 Moon1000 数据集

构建一个二分类数据集 **Moon1000**：1000 条样本，每个样本有 2 个特征。本数据集的样本来自带噪声的两个弯月形状函数，每个弯月对应一个类别。


In [ ]:
import math


def make_moons(n_samples=1000, shuffle=True, noise=None, seed=None):
    """生成两个弯月形状的二分类数据。"""
    g = torch.Generator().manual_seed(seed) if seed is not None else None
    n_out = n_samples // 2
    n_in  = n_samples - n_out

    # 外圈：第 0 类
    outer_x = torch.cos(torch.linspace(0, math.pi, n_out))
    outer_y = torch.sin(torch.linspace(0, math.pi, n_out))
    # 内圈：第 1 类（朝下偏移、左移构成弯月）
    inner_x = 1 - torch.cos(torch.linspace(0, math.pi, n_in))
    inner_y = 0.5 - torch.sin(torch.linspace(0, math.pi, n_in))

    X = torch.stack([
        torch.cat([outer_x, inner_x]),
        torch.cat([outer_y, inner_y]),
    ], dim=1)
    y = torch.cat([torch.zeros(n_out), torch.ones(n_in)])

    if shuffle:
        idx = torch.randperm(X.shape[0], generator=g)
        X = X[idx]; y = y[idx]
    if noise is not None:
        X = X + torch.normal(mean=0.0, std=noise, size=X.shape, generator=g)
    return X, y


torch.manual_seed(0)
n_samples = 1000
X, y = make_moons(n_samples=n_samples, shuffle=True, noise=0.2)

plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0].tolist(), X[:, 1].tolist(), marker='*', c=y.tolist())
plt.title('Moon1000'); plt.show()


将 1000 条样本数据拆分成训练集、验证集和测试集，其中训练集 640 条、验证集 160 条、测试集 200 条。


In [ ]:
num_train, num_dev, num_test = 640, 160, 200

X_train, y_train = X[:num_train], y[:num_train]
X_dev,   y_dev   = X[num_train:num_train + num_dev], y[num_train:num_train + num_dev]
X_test,  y_test  = X[num_train + num_dev:], y[num_train + num_dev:]

y_train = y_train.reshape(-1, 1)
y_dev   = y_dev.reshape(-1, 1)
y_test  = y_test.reshape(-1, 1)

print('X_train shape:', X_train.shape, 'y_train shape:', y_train.shape)


## 3. 模型构建：`Model_LR`

Logistic 回归模型就是**线性层 + Logistic 函数**的组合，权重和偏置通常初始化为 0。批量计算公式为

$$ \hat{\boldsymbol{y}} = p(\boldsymbol{y}\mid \boldsymbol{x}) = \sigma(\boldsymbol{X}\boldsymbol{w}+b), $$

其中 $\boldsymbol{X}\in\mathbb{R}^{N\times D}$ 为 $N$ 个样本的特征矩阵。

延续第 1、2 章的 `Op` 接口风格自己实现一个 `Model_LR`，方便后面手动写反向函数。


In [ ]:
class Op:
    """算子基类。"""
    def __call__(self, inputs):
        return self.forward(inputs)
    def forward(self, *args, **kwargs):
        raise NotImplementedError
    def backward(self, *args, **kwargs):
        raise NotImplementedError


class Model_LR(Op):
    def __init__(self, input_size):
        super().__init__()
        # 权重和偏置全 0 初始化
        self.params = {
            'w': torch.zeros(input_size, 1),
            'b': torch.zeros(1),
        }
        # 存放参数的梯度
        self.grads = {}
        self.X = None
        self.outputs = None

    def forward(self, inputs):
        """输入 X (N, D)，输出预测概率 (N, 1)。"""
        self.X = inputs
        score = inputs @ self.params['w'] + self.params['b']
        self.outputs = logistic(score)
        return self.outputs

    def backward(self, labels):
        """直接给出 loss 关于参数的梯度（手动推导）。"""
        N = labels.shape[0]
        self.grads['w'] = -1 / N * (self.X.t() @ (labels - self.outputs))
        self.grads['b'] = -1 / N * (labels - self.outputs).sum()


# 随机生成 3 条长度为 4 的数据测试一下
torch.manual_seed(0)
inputs = torch.randn(3, 4)
print('Input:\n', inputs)

model = Model_LR(input_size=4)
outputs = model(inputs)
print('Output:\n', outputs)


从输出结果看，模型最终的输出 $g(\cdot)$ 恒为 0.5。这是由于全 0 初始化后，不论输入值大小，Logistic 函数的输入恒为 0，所以输出恒为 $\sigma(0)=0.5$。


## 4. 损失函数：二分类交叉熵

给定 $N$ 个训练样本，使用交叉熵损失，Logistic 回归的风险函数为

$$ \mathcal{R}(\boldsymbol{w},b)=-\frac{1}{N}\sum_{n=1}^N \Big(y^{(n)}\log\hat y^{(n)} + (1-y^{(n)})\log(1-\hat y^{(n)})\Big). $$

向量形式：

$$ \mathcal{R}(\boldsymbol{w},b)=-\frac{1}{N}\Big(\boldsymbol{y}^\top \log\hat{\boldsymbol{y}} + (1-\boldsymbol{y})^\top\log(1-\hat{\boldsymbol{y}})\Big). $$


In [ ]:
class BinaryCrossEntropyLoss(Op):
    def __init__(self):
        self.predicts = None
        self.labels = None

    def __call__(self, predicts, labels):
        return self.forward(predicts, labels)

    def forward(self, predicts, labels):
        self.predicts = predicts
        self.labels = labels
        N = predicts.shape[0]
        # 为数值稳定，把 predicts clip 到 (eps, 1-eps)
        eps = 1e-9
        p = predicts.clamp(eps, 1 - eps)
        loss = -1 / N * (labels.t() @ torch.log(p) + (1 - labels).t() @ torch.log(1 - p))
        return loss.squeeze()


# 测试一下：把标签全设为 1，预测全是 0.5，loss = -log(0.5) ≈ 0.693
labels = torch.ones(3, 1)
bce_loss = BinaryCrossEntropyLoss()
print('loss:', bce_loss(outputs, labels))


## 5. 模型优化：梯度下降法

不同于线性回归可以直接使用最小二乘法，Logistic 回归需要用**优化算法**对参数进行有限次迭代来获取更优模型。这里使用最简单常用的**梯度下降法**。

### 5.1 梯度计算

风险函数 $\mathcal{R}(\boldsymbol{w},b)$ 关于参数 $\boldsymbol{w}$ 和 $b$ 的偏导数为

$$ \frac{\partial \mathcal{R}}{\partial \boldsymbol{w}} = -\frac{1}{N}\boldsymbol{X}^\top(\boldsymbol{y}-\hat{\boldsymbol{y}}), \qquad \frac{\partial \mathcal{R}}{\partial b} = -\frac{1}{N}\mathbf{1}^\top(\boldsymbol{y}-\hat{\boldsymbol{y}}). $$

我们已经在 `Model_LR.backward()` 里实现了这两个偏导数。**注意**：这里 `backward` 实现的是"损失对参数"的梯度，不是 forward 算子本身的梯度——我们手动把整个 Loss + Sigmoid + Linear 的复合梯度推导出来直接使用。

### 5.2 优化器基类与 `SimpleBatchGD`

参数更新公式：$\boldsymbol{w} \leftarrow \boldsymbol{w} - \alpha \frac{\partial \mathcal{R}}{\partial \boldsymbol{w}}$。

我们先定义优化器基类 `Optimizer`，再实现简单批量梯度下降 `SimpleBatchGD`。


In [ ]:
from abc import ABC, abstractmethod


class Optimizer(ABC):
    """优化器基类。"""

    def __init__(self, init_lr, model):
        self.init_lr = init_lr
        self.model = model

    @abstractmethod
    def step(self):
        pass


class SimpleBatchGD(Optimizer):
    """全批量梯度下降。"""

    def step(self):
        if isinstance(self.model.params, dict):
            for key in self.model.params.keys():
                self.model.params[key] = self.model.params[key] - self.init_lr * self.model.grads[key]


## 6. 评价指标：准确率

在分类任务中，通常使用**准确率**（Accuracy）作为评价指标——正确预测的样本数与总样本数的比值：

$$ \mathcal{A} = \frac{1}{N}\mathbf{1}^\top I(\boldsymbol{y}=\hat{\boldsymbol{y}}). $$


In [ ]:
def accuracy(preds, labels):
    """
    - preds (N, 1) 二分类或 (N, C) 多分类
    - labels (N, 1) 或 (N,)
    """
    if preds.dim() > 1 and preds.shape[1] == 1:
        # 二分类：阈值 0.5
        preds = (preds >= 0.5).float()
    elif preds.dim() > 1:
        # 多分类：取 argmax
        preds = preds.argmax(dim=1)
    labels = labels.reshape(preds.shape)
    return (preds == labels).float().mean()


# 测试一下
p = torch.tensor([[0.], [1.], [1.], [0.]])
l = torch.tensor([[1.], [1.], [0.], [0.]])
print('accuracy:', accuracy(p, l).item())   # 2/4 = 0.5


## 7. 完善 Runner：`RunnerV2`

第 2 章实现的 `RunnerV1` 用于直接计算解析解的情况；本章采用梯度下降法，所以需要在 `train` 函数中加入梯度下降的迭代过程。同时基于**早停法**的思想：在训练过程中引入验证集，计算验证集上的损失和评价指标，保存当前最优模型。

> **早停法**（Early Stopping）：拟合能力非常强的模型在长时间训练后容易过拟合。在模型优化时使用验证集上的错误代替期望错误，当验证集错误率不再下降就停止迭代，是一种简单有效的正则化方法。

> **提示**：本书把 `RunnerV2` 也放在 `nndl/runner.py` 里；后续章节可以直接 `from nndl.runner import RunnerV2` 复用，不必再粘一遍。

In [ ]:
import os


class RunnerV2:
    """全批量梯度下降；记录 train/dev 历史到 history 字典；保存 dev 最优模型。

    - model：自定义 Op 算子（持有 params/grads 字典并实现 backward(labels)）
    - optimizer：实现 step() 的对象（如 SimpleBatchGD）
    - metric / loss_fn：评价指标与损失函数（返回 tensor 或 float 均可）
    """

    def __init__(self, model, optimizer, metric, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.metric = metric
        # 训练过程指标统一收纳到 history 字典中
        self.history = {
            "train_loss": [], "dev_loss": [],
            "train_score": [], "dev_score": [],
        }

    def train(self, train_set, dev_set, num_epochs=100, log_epochs=100,
              save_path="model_best.pt"):
        # 用 -inf 初始化保证首轮一定触发对比
        best_score = -float("inf")
        X, y = train_set
        for epoch in range(num_epochs):
            # 训练一步
            logits = self.model(X)
            trn_loss = self.loss_fn(logits, y)
            if hasattr(trn_loss, "item"):
                trn_loss = trn_loss.item()
            trn_score = self.metric(logits, y)
            if hasattr(trn_score, "item"):
                trn_score = trn_score.item()
            self.model.backward(y)
            self.optimizer.step()
            self.history["train_loss"].append(trn_loss)
            self.history["train_score"].append(trn_score)
            # 验证集评估 + 写入历史 + best-checkpoint
            dev_score, dev_loss = self.evaluate(dev_set)
            self.history["dev_loss"].append(dev_loss)
            self.history["dev_score"].append(dev_score)
            if dev_score > best_score:
                self.save_model(save_path)
                print(f"best score updated: {best_score:.5f} -> {dev_score:.5f}")
                best_score = dev_score
            if (epoch + 1) % log_epochs == 0:
                print(f"[Train] epoch {epoch+1}  loss {trn_loss:.4f}  score {trn_score:.4f}")
                print(f"[Dev]   epoch {epoch+1}  loss {dev_loss:.4f}  score {dev_score:.4f}")

    def evaluate(self, data_set):
        """纯查询：不写入历史。
        训练循环里由 train 自己 append，最终测试集评估不会污染 dev 历史。
        """
        X, y = data_set
        logits = self.model(X)
        loss = self.loss_fn(logits, y)
        if hasattr(loss, "item"):
            loss = loss.item()
        score = self.metric(logits, y)
        if hasattr(score, "item"):
            score = score.item()
        return score, loss

    def predict(self, X):
        return self.model(X)

    def save_model(self, save_path):
        os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
        torch.save(self.model.params, save_path)

    def load_model(self, save_path):
        self.model.params = torch.load(save_path, weights_only=True)


## 8. 模型训练

使用训练集和验证集进行模型训练，共训练 500 个回合，每隔 50 个回合打印一次。


In [ ]:
input_size = 2
model = Model_LR(input_size)
optimizer = SimpleBatchGD(init_lr=0.1, model=model)
loss_fn   = BinaryCrossEntropyLoss()
metric    = accuracy

runner = RunnerV2(model, optimizer, metric, loss_fn)
runner.train([X_train, y_train], [X_dev, y_dev],
             num_epochs=500, log_epochs=100,
             save_path='./checkpoint/model_lr_best.pt')


### 训练过程可视化


In [ ]:
def plot(runner):
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.plot(runner.history['train_score'], color='red',  label='Train accuracy')
    plt.plot(runner.history['dev_score'],   color='blue', label='Dev accuracy')
    plt.ylabel('score'); plt.xlabel('epoch'); plt.legend(loc='lower right')

    plt.subplot(1, 2, 2)
    plt.plot(runner.history['train_loss'], color='red',  label='Train loss')
    plt.plot(runner.history['dev_loss'],   color='blue', label='Dev loss')
    plt.ylabel('loss'); plt.xlabel('epoch'); plt.legend(loc='upper right')

    plt.tight_layout(); plt.show()


plot(runner)


## 9. 模型评价


In [ ]:
score, loss = runner.evaluate([X_test, y_test])
print(f'[Test] score / loss: {score:.4f} / {loss:.4f}')


### 决策边界可视化


In [ ]:
with torch.no_grad():
    xx, yy = torch.meshgrid(
        torch.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200),
        torch.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 200),
        indexing='xy',
    )
    grid = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
    prob = model(grid).reshape(xx.shape)

plt.figure(figsize=(5, 5))
plt.contourf(xx.numpy(), yy.numpy(), prob.numpy(), levels=20, alpha=0.4, cmap='coolwarm')
plt.scatter(X[:, 0], X[:, 1], marker='*', c=y, s=10)
plt.title('Logistic regression decision boundary'); plt.show()


**\\\\练一练\\\\**：尝试调整学习率和训练回合数等超参数，观察训练过程，并努力得到更高的准确率。

下一节我们把 Logistic 回归推广到多分类——Softmax 回归，并跑通经典的鸢尾花分类任务。
